In [2]:
# Define the query
query = """
SELECT *
FROM [dbo].[tbl_errordetails_datamodel]
where file_id > ISNULL((SELECT max(FILE_ID)
        FROM [dbo].[tbl_error_count_by_attribute_datamodel_updated]), 0);
"""

# Execute the query
cursor.execute(query)

# Fetch all results
rows = cursor.fetchall()

# Convert result to DataFrame
columns = [column[0] for column in cursor.description]

# Convert pyodbc.Row objects to list of tuples
rows = [tuple(row) for row in rows]

# Define schema explicitly based on the columns
schema = StructType([StructField(col, StringType(), True) for col in columns])

# Convert result to Spark DataFrame
if rows:
    df = spark.createDataFrame(rows, schema)
else:
    df = spark.createDataFrame([], schema)

df.createOrReplaceTempView('vw_tbl_errordetails_datamodel')

In [5]:
def sp_error_by_attribute_and_attributegroup_datamodel_updated():
    tbl_error_count_by_attribute_datamodel_updated = spark.sql(""" SELECT DISTINCT 
            file_id as file_id,
            attribute_group AS attribute_group, 
            business_attribute_name AS business_attribute_name,
            preferred_asset AS preferred_asset,
            error_id AS error_id,
            dimension AS dimension 
            FROM (
                SELECT DISTINCT 
                file_id, 
                attribute_group, 
                business_attribute_name,
                'Default' as preferred_asset,
	            error_id,
                dimension 
                from vw_tbl_errordetails_datamodel)
    """)
    
    # tbl_error_count_by_attribute_datamodel_updated.createOrReplaceTempView('tbl_error_count_by_attribute_datamodel_updated')
    tbl_error_count_by_attribute_datamodel_updated.toPandas().to_sql('tbl_error_count_by_attribute_datamodel_updated', con=engine, if_exists='append', index=False)

In [ ]:
sp_error_by_attribute_and_attributegroup_datamodel_updated()